# Prepare shared Stage 2 plans

Coordinator entry point. Read `docs/stage2_interfaces.md` at the published base.
Mount the intended Drive account in Colab before setup. This notebook does not mount,
authenticate to Hugging Face, install GPU packages, load weights, fit heads or run workers.
Install the CPU environment with `pip install -r requirements-stage2-cpu.txt` if needed;
record actual versions. Never update the checkout midway through a run.
Worker notebooks consume the printed plan; per-runtime Gemma authentication remains required
in model-loading notebooks. A passed plan describes readiness and inventory, not arrays.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO = Path('/content/Research')  # Clean checkout at the published immutable base.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
if not (REPO / 'src/stage2/interfaces.py').is_file():
    raise ValueError('Set REPO to the shared-foundation checkout before continuing')
if not PHASE1_ROOT.is_dir():
    raise ValueError('Mount the intended Drive account and set PHASE1_ROOT')
os.chdir(REPO)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
if subprocess.check_output(['git','status','--porcelain'], text=True).strip():
    raise ValueError('Use a clean frozen checkout; do not pull during a job')
print(json.dumps({'repo': str(REPO), 'phase1_root': str(PHASE1_ROOT), 'code_commit': CODE_COMMIT}, indent=2))


In [ ]:
# Rerun this cell to replace settings; execution always starts a fresh verifier process.
# First run: shared plan, with no model bindings, is sufficient for E03.
# Later runs: set SHARED_PLAN to that printed manifest to add model bindings.
SHARED_PLAN = None
DATASET_MANIFEST = PHASE1_ROOT / 'shared/8cf019d18a28a8bb18540f4ce2e26b17fc0af15a12c8d016138dc99c397c855a/manifests/54825d236efcd88cc5252918ebce6d1dad6109ef10a244281f262204bc6cc5bd/_COMPLETE.json'
DATA_VALIDATION = None  # Typed deterministic discovery in this dataset/protocol namespace.
BINDINGS = {}
# Optional, bind once here (no weights are loaded):
# BINDINGS = {'qwen': {'configuration_manifest': '/.../_COMPLETE.json',
#                      'model_validation': None, 'smoke': None}}
# None evidence pointers trigger typed discovery in the exact model namespace.
# Use 'gemma' for the other model. Never enter HF tokens in this mapping.
STORAGE_SCOPE = 'persistent-drive'


In [ ]:
import tempfile
from src.stage2.schema import marker_path
if SHARED_PLAN is not None:
    marker_path(SHARED_PLAN, PHASE1_ROOT)
else:
    marker_path(DATASET_MANIFEST, PHASE1_ROOT)
if DATA_VALIDATION is not None:
    marker_path(DATA_VALIDATION, PHASE1_ROOT)
for model, binding in BINDINGS.items():
    if model not in ('qwen','gemma'):
        raise ValueError('Unknown MODEL binding')
    for key, value in binding.items():
        if value is not None:
            p = marker_path(value, PHASE1_ROOT)
            if not p.is_file(): raise ValueError('Missing supplied binding: '+key)
print(json.dumps({'shared_plan':str(SHARED_PLAN), 'dataset_manifest':str(DATASET_MANIFEST),
                  'data_validation':str(DATA_VALIDATION), 'bindings':BINDINGS},indent=2))
command=[sys.executable,'-m','src.stage2.operations','prepare',
         '--phase1-root',str(PHASE1_ROOT),'--storage-scope',STORAGE_SCOPE]
if SHARED_PLAN is not None: command += ['--manifest',str(SHARED_PLAN)]
else: command += ['--dataset-manifest',str(DATASET_MANIFEST)]
if DATA_VALIDATION is not None: command += ['--data-validation',str(DATA_VALIDATION)]
with tempfile.TemporaryDirectory() as temporary:
    settings=Path(temporary)/'bindings.json'
    settings.write_text(json.dumps(BINDINGS),encoding='utf-8')
    completed=subprocess.run(command+['--bindings',str(settings)],text=True,capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError('Stage 2 preparation blocked; inspect the diagnostic')
